## Imports y Configuración

Define el video, las rutas y el tamaño principal de cada bloque.


In [1]:
from pathlib import Path
import math
import shutil

import cv2
import numpy as np

VIDEO = Path("Videos") / "video30min-11to22.mp4"
SALIDA = Path("Videos")

FPS_SALIDA = 5.0
SEGUNDO_INICIO = None
SEGUNDO_FIN = None

FRAMES_PRINCIPALES_POR_BLOQUE = 500
SOLAPAMIENTO_TOTAL = 20

GUARDAR_VISTAS = True
GUARDAR_DIAGNOSTICOS = True
LIMPIAR_ANTERIORES = True
COMPRESION_PNG = 3


## Extracción de Frames

Extrae todos los frames del video y conserva nombres consecutivos.


In [2]:
def limpiar_png(carpeta):
    carpeta = Path(carpeta)
    if carpeta.exists():
        for archivo in carpeta.glob("*.png"):
            archivo.unlink()


def guardar_png(ruta, imagen):
    ruta = Path(ruta)
    ruta.parent.mkdir(parents=True, exist_ok=True)

    correcto = cv2.imwrite(
        str(ruta),
        imagen,
        [cv2.IMWRITE_PNG_COMPRESSION, COMPRESION_PNG],
    )

    if not correcto:
        raise RuntimeError(f"No se pudo guardar {ruta}")


def extraer_frames(
    video=VIDEO,
    salida=SALIDA,
    fps_salida=FPS_SALIDA,
    segundo_inicio=SEGUNDO_INICIO,
    segundo_fin=SEGUNDO_FIN,
):
    video = Path(video)
    salida = Path(salida)

    if not video.is_file():
        raise FileNotFoundError(video)

    captura = cv2.VideoCapture(str(video))
    fps_video = float(captura.get(cv2.CAP_PROP_FPS))
    total_video = int(captura.get(cv2.CAP_PROP_FRAME_COUNT))

    if fps_video <= 0 or total_video <= 0:
        captura.release()
        raise RuntimeError("No se pudieron leer los datos del video")

    duracion = total_video / fps_video
    inicio = 0.0 if segundo_inicio is None else max(0.0, float(segundo_inicio))
    fin = duracion if segundo_fin is None else min(duracion, float(segundo_fin))

    if fin <= inicio:
        captura.release()
        raise ValueError("El intervalo indicado no es válido")

    tiempos = np.arange(inicio, fin, 1.0 / fps_salida, dtype=np.float64)
    base = salida / video.stem
    carpeta_frames = base / "frames"
    carpeta_frames.mkdir(parents=True, exist_ok=True)

    if LIMPIAR_ANTERIORES:
        limpiar_png(carpeta_frames)

    digitos = max(4, len(str(max(0, len(tiempos) - 1))))
    guardados = 0

    for indice, tiempo in enumerate(tiempos):
        captura.set(cv2.CAP_PROP_POS_MSEC, float(tiempo) * 1000.0)
        correcto, frame = captura.read()

        if not correcto:
            break

        nombre = f"{indice:0{digitos}d}.png"
        guardar_png(carpeta_frames / nombre, frame)
        guardados += 1

        if guardados % 100 == 0:
            print(f"Frames: {guardados}/{len(tiempos)}")

    captura.release()

    print(f"Frames extraídos: {guardados}")
    print(f"Carpeta: {carpeta_frames}")
    return base


## Configuración del HUD

Define los rangos de verde y la cobertura de cada elemento.


In [3]:
VERDE_FUERTE = (37, 90, 105, 55, 15, 12)
VERDE_MEDIO = (32, 100, 38, 30, 6, 5)
VERDE_TENUE = (28, 108, 16, 16, 2, 2)

MUESTRAS_ATLAS = 400
UMBRAL_ATLAS = 0.12
RADIO_ATLAS = 6

DX_INICIAL = 0.088
DY_INICIAL = 0.082
DX_MINIMO = 0.055
DX_MAXIMO = 0.165
DY_MINIMO = 0.045
DY_MAXIMO = 0.170

GROSOR_CROSSHAIR = 8
RADIO_RESIDUAL_CROSSHAIR = 14
RADIO_RESIDUAL_HUD = 11

GROSOR_CIRCULO = 9
MARGEN_CIRCULO = 4


## Detección del HUD

Detecta la fluorescencia únicamente dentro de las zonas esperadas.


In [4]:
def detectar_verde(frame, configuracion):
    h_min, h_max, s_min, v_min, diferencia_rojo, diferencia_azul = configuracion
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    azul, verde, rojo = cv2.split(frame)

    azul = azul.astype(np.int16)
    verde = verde.astype(np.int16)
    rojo = rojo.astype(np.int16)

    rango = cv2.inRange(
        hsv,
        np.array([h_min, s_min, v_min], dtype=np.uint8),
        np.array([h_max, 255, 255], dtype=np.uint8),
    )

    dominio = (
        (verde >= rojo + diferencia_rojo)
        & (verde >= azul + diferencia_azul)
    ).astype(np.uint8) * 255

    return cv2.bitwise_and(rango, dominio)


def crear_zona(forma, cajas):
    alto, ancho = forma
    zona = np.zeros(forma, dtype=np.uint8)

    for x1, y1, x2, y2 in cajas:
        izquierda = int(np.clip(round(ancho * x1), 0, ancho - 1))
        superior = int(np.clip(round(alto * y1), 0, alto - 1))
        derecha = int(np.clip(round(ancho * x2), 0, ancho - 1))
        inferior = int(np.clip(round(alto * y2), 0, alto - 1))
        cv2.rectangle(zona, (izquierda, superior), (derecha, inferior), 255, -1)

    return zona


def obtener_zonas(forma):
    texto = crear_zona(forma, [
        (0.015, 0.015, 0.270, 0.160),
        (0.015, 0.170, 0.105, 0.390),
        (0.730, 0.015, 0.990, 0.165),
        (0.010, 0.615, 0.085, 0.740),
        (0.010, 0.780, 0.270, 0.995),
        (0.845, 0.750, 0.995, 0.995),
    ])

    brujula = crear_zona(forma, [(0.380, 0.000, 0.620, 0.165)])
    crosshair = crear_zona(forma, [(0.330, 0.270, 0.670, 0.765)])

    inferior = crear_zona(forma, [
        (0.005, 0.770, 0.275, 0.995),
        (0.340, 0.770, 0.660, 0.995),
        (0.840, 0.745, 0.995, 0.995),
    ])

    return texto, brujula, crosshair, inferior


def expandir(mascara, radio):
    tamano = 2 * int(radio) + 1
    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (tamano, tamano),
    )
    return cv2.dilate(mascara, kernel)


## Máscara del Crosshair

Busca una geometría simétrica y cubre los residuos cercanos.


In [5]:
# Crosshair

DX_INICIAL = 0.080
DY_INICIAL = 0.082

DX_MINIMO = 0.060
DX_MAXIMO = 0.125

DY_MINIMO = 0.055
DY_MAXIMO = 0.145

PASO_BUSQUEDA_CROSSHAIR = 3
PESO_APERTURA_ANTERIOR = 0.82
CAMBIO_MAXIMO_X = 24
CAMBIO_MAXIMO_Y = 18

GROSOR_MEDICION = 5
GROSOR_CROSSHAIR = 9
MARGEN_CROSSHAIR = 3
RADIO_RESIDUAL_CROSSHAIR = 20

PATA_HORIZONTAL_RELATIVA = 0.028
PATA_VERTICAL_RELATIVA = 0.042

BRAZO_HORIZONTAL_RELATIVO = 0.76
BRAZO_VERTICAL_RELATIVO = 0.58

RADIO_X_CENTRAL = 12

POSICION_MARCA_LATERAL = 0.78
LARGO_MARCA_LATERAL = 0.18


# Indicadores circulares inferiores

RADIO_CIRCULO_MINIMO = 0.018
RADIO_CIRCULO_MAXIMO = 0.050

GROSOR_CIRCULO = 10
MARGEN_CIRCULO = 4

PESO_CIRCULO_ANTERIOR = 0.85
MINIMO_PIXELES_ARCO = 14

RADIO_LINEAS_CIRCULO = 18
DILATACION_LINEAS_CIRCULO = 5

In [6]:
def obtener_evidencia_crosshair(
    frame,
    zona_crosshair,
):
    fuerte = cv2.bitwise_and(
        detectar_verde(
            frame,
            VERDE_FUERTE,
        ),
        zona_crosshair,
    )

    medio = cv2.bitwise_and(
        detectar_verde(
            frame,
            VERDE_MEDIO,
        ),
        zona_crosshair,
    )

    tenue = cv2.bitwise_and(
        detectar_verde(
            frame,
            VERDE_TENUE,
        ),
        zona_crosshair,
    )

    evidencia = cv2.bitwise_or(
        fuerte,
        medio,
    )

    evidencia_tenue = cv2.bitwise_and(
        tenue,
        expandir(
            evidencia,
            5,
        ),
    )

    evidencia = cv2.bitwise_or(
        evidencia,
        evidencia_tenue,
    )

    evidencia = cv2.morphologyEx(
        evidencia,
        cv2.MORPH_CLOSE,
        cv2.getStructuringElement(
            cv2.MORPH_CROSS,
            (3, 3),
        ),
        iterations=1,
    )

    return evidencia


def dibujar_esquinas_medicion(
    forma,
    distancia_x,
    distancia_y,
):
    alto, ancho = forma

    centro_x = ancho // 2
    centro_y = alto // 2

    mascara = np.zeros(
        forma,
        dtype=np.uint8,
    )

    largo_horizontal = max(
        18,
        int(
            ancho
            * PATA_HORIZONTAL_RELATIVA
        ),
    )

    largo_vertical = max(
        18,
        int(
            alto
            * PATA_VERTICAL_RELATIVA
        ),
    )

    for signo_x, signo_y in [
        (-1, -1),
        (1, -1),
        (-1, 1),
        (1, 1),
    ]:
        x = (
            centro_x
            + signo_x * distancia_x
        )

        y = (
            centro_y
            + signo_y * distancia_y
        )

        cv2.line(
            mascara,
            (x, y),
            (
                x - signo_x * largo_horizontal,
                y,
            ),
            255,
            GROSOR_MEDICION,
            cv2.LINE_AA,
        )

        cv2.line(
            mascara,
            (x, y),
            (
                x,
                y - signo_y * largo_vertical,
            ),
            255,
            GROSOR_MEDICION,
            cv2.LINE_AA,
        )

    return mascara


def puntuar_esquinas_crosshair(
    evidencia,
    distancia_x,
    distancia_y,
):
    alto, ancho = evidencia.shape

    centro_x = ancho // 2
    centro_y = alto // 2

    largo_horizontal = max(
        18,
        int(
            ancho
            * PATA_HORIZONTAL_RELATIVA
        ),
    )

    largo_vertical = max(
        18,
        int(
            alto
            * PATA_VERTICAL_RELATIVA
        ),
    )

    soporte_cercano = expandir(
        evidencia,
        3,
    )

    puntuaciones = []

    for signo_x, signo_y in [
        (-1, -1),
        (1, -1),
        (-1, 1),
        (1, 1),
    ]:
        x = (
            centro_x
            + signo_x * distancia_x
        )

        y = (
            centro_y
            + signo_y * distancia_y
        )

        mascara_horizontal = np.zeros_like(
            evidencia
        )

        mascara_vertical = np.zeros_like(
            evidencia
        )

        cv2.line(
            mascara_horizontal,
            (x, y),
            (
                x - signo_x * largo_horizontal,
                y,
            ),
            255,
            GROSOR_MEDICION,
            cv2.LINE_AA,
        )

        cv2.line(
            mascara_vertical,
            (x, y),
            (
                x,
                y - signo_y * largo_vertical,
            ),
            255,
            GROSOR_MEDICION,
            cv2.LINE_AA,
        )

        pixeles_horizontal = (
            mascara_horizontal > 0
        )

        pixeles_vertical = (
            mascara_vertical > 0
        )

        total_horizontal = max(
            1,
            np.count_nonzero(
                pixeles_horizontal
            ),
        )

        total_vertical = max(
            1,
            np.count_nonzero(
                pixeles_vertical
            ),
        )

        apoyo_horizontal = (
            np.count_nonzero(
                pixeles_horizontal
                & (soporte_cercano > 0)
            )
            / total_horizontal
        )

        apoyo_vertical = (
            np.count_nonzero(
                pixeles_vertical
                & (soporte_cercano > 0)
            )
            / total_vertical
        )

        apoyo_esquina = min(
            apoyo_horizontal,
            apoyo_vertical,
        )

        puntuaciones.append(
            apoyo_esquina
        )

    esquinas_buenas = sum(
        puntuacion >= 0.18
        for puntuacion in puntuaciones
    )

    esquinas_fuertes = sum(
        puntuacion >= 0.35
        for puntuacion in puntuaciones
    )

    mediana = float(
        np.median(
            puntuaciones
        )
    )

    minimo = float(
        np.min(
            puntuaciones
        )
    )

    puntuacion_total = (
        mediana * 220.0
        + minimo * 50.0
        + esquinas_buenas * 30.0
        + esquinas_fuertes * 20.0
    )

    if esquinas_buenas == 0:
        puntuacion_total -= 220.0

    elif esquinas_buenas == 1:
        puntuacion_total -= 130.0

    elif esquinas_buenas == 2:
        puntuacion_total -= 35.0

    return (
        puntuacion_total,
        esquinas_buenas,
        puntuaciones,
    )


def estimar_apertura_crosshair(
    evidencia,
    apertura_anterior=None,
):
    alto, ancho = evidencia.shape

    if apertura_anterior is None:
        apertura_anterior = (
            int(
                ancho
                * DX_INICIAL
            ),
            int(
                alto
                * DY_INICIAL
            ),
        )

    anterior_x, anterior_y = (
        apertura_anterior
    )

    minimo_x = int(
        ancho
        * DX_MINIMO
    )

    maximo_x = int(
        ancho
        * DX_MAXIMO
    )

    minimo_y = int(
        alto
        * DY_MINIMO
    )

    maximo_y = int(
        alto
        * DY_MAXIMO
    )

    (
        puntuacion_anterior,
        esquinas_anteriores,
        _,
    ) = puntuar_esquinas_crosshair(
        evidencia,
        anterior_x,
        anterior_y,
    )

    mejor_apertura = (
        anterior_x,
        anterior_y,
    )

    mejor_puntuacion = (
        puntuacion_anterior
    )

    mejores_esquinas = (
        esquinas_anteriores
    )

    for distancia_y in range(
        minimo_y,
        maximo_y + 1,
        PASO_BUSQUEDA_CROSSHAIR,
    ):
        for distancia_x in range(
            minimo_x,
            maximo_x + 1,
            PASO_BUSQUEDA_CROSSHAIR,
        ):
            (
                puntuacion,
                esquinas_buenas,
                _,
            ) = puntuar_esquinas_crosshair(
                evidencia,
                distancia_x,
                distancia_y,
            )

            cambio = (
                abs(
                    distancia_x
                    - anterior_x
                )
                + abs(
                    distancia_y
                    - anterior_y
                )
            )

            puntuacion -= (
                cambio
                * 0.06
            )

            if puntuacion > mejor_puntuacion:
                mejor_puntuacion = (
                    puntuacion
                )

                mejor_apertura = (
                    distancia_x,
                    distancia_y,
                )

                mejores_esquinas = (
                    esquinas_buenas
                )

    if mejores_esquinas < 2:
        return (
            apertura_anterior,
            esquinas_anteriores,
            puntuacion_anterior,
        )

    objetivo_x = int(
        np.clip(
            mejor_apertura[0],
            anterior_x - CAMBIO_MAXIMO_X,
            anterior_x + CAMBIO_MAXIMO_X,
        )
    )

    objetivo_y = int(
        np.clip(
            mejor_apertura[1],
            anterior_y - CAMBIO_MAXIMO_Y,
            anterior_y + CAMBIO_MAXIMO_Y,
        )
    )

    nueva_x = int(round(
        PESO_APERTURA_ANTERIOR
        * anterior_x
        + (
            1.0
            - PESO_APERTURA_ANTERIOR
        )
        * objetivo_x
    ))

    nueva_y = int(round(
        PESO_APERTURA_ANTERIOR
        * anterior_y
        + (
            1.0
            - PESO_APERTURA_ANTERIOR
        )
        * objetivo_y
    ))

    nueva_x = int(
        round(
            nueva_x / 2
        )
        * 2
    )

    nueva_y = int(
        round(
            nueva_y / 2
        )
        * 2
    )

    return (
        (
            nueva_x,
            nueva_y,
        ),
        mejores_esquinas,
        mejor_puntuacion,
    )


In [7]:
def dibujar_crosshair_completo(
    forma,
    apertura,
):
    alto, ancho = forma

    centro_x = ancho // 2
    centro_y = alto // 2

    distancia_x, distancia_y = (
        apertura
    )

    mascara = np.zeros(
        forma,
        dtype=np.uint8,
    )

    largo_horizontal = max(
        20,
        int(
            ancho
            * PATA_HORIZONTAL_RELATIVA
        ),
    )

    largo_vertical = max(
        20,
        int(
            alto
            * PATA_VERTICAL_RELATIVA
        ),
    )

    for signo_x, signo_y in [
        (-1, -1),
        (1, -1),
        (-1, 1),
        (1, 1),
    ]:
        x = (
            centro_x
            + signo_x * distancia_x
        )

        y = (
            centro_y
            + signo_y * distancia_y
        )

        cv2.line(
            mascara,
            (x, y),
            (
                x - signo_x * largo_horizontal,
                y,
            ),
            255,
            GROSOR_CROSSHAIR,
            cv2.LINE_AA,
        )

        cv2.line(
            mascara,
            (x, y),
            (
                x,
                y - signo_y * largo_vertical,
            ),
            255,
            GROSOR_CROSSHAIR,
            cv2.LINE_AA,
        )

    radio_x = RADIO_X_CENTRAL

    cv2.line(
        mascara,
        (
            centro_x - radio_x,
            centro_y - radio_x,
        ),
        (
            centro_x + radio_x,
            centro_y + radio_x,
        ),
        255,
        GROSOR_CROSSHAIR,
        cv2.LINE_AA,
    )

    cv2.line(
        mascara,
        (
            centro_x - radio_x,
            centro_y + radio_x,
        ),
        (
            centro_x + radio_x,
            centro_y - radio_x,
        ),
        255,
        GROSOR_CROSSHAIR,
        cv2.LINE_AA,
    )

    cv2.circle(
        mascara,
        (
            centro_x,
            centro_y,
        ),
        4,
        255,
        -1,
        cv2.LINE_AA,
    )

    brazo_horizontal = max(
        22,
        int(
            distancia_x
            * BRAZO_HORIZONTAL_RELATIVO
        ),
    )

    brazo_vertical = max(
        14,
        int(
            distancia_y
            * BRAZO_VERTICAL_RELATIVO
        ),
    )

    cv2.line(
        mascara,
        (
            centro_x - brazo_horizontal,
            centro_y,
        ),
        (
            centro_x + brazo_horizontal,
            centro_y,
        ),
        255,
        GROSOR_CROSSHAIR,
        cv2.LINE_AA,
    )

    cv2.line(
        mascara,
        (
            centro_x,
            centro_y - brazo_vertical,
        ),
        (
            centro_x,
            centro_y + brazo_vertical,
        ),
        255,
        GROSOR_CROSSHAIR,
        cv2.LINE_AA,
    )

    desplazamiento = int(
        distancia_x
        * POSICION_MARCA_LATERAL
    )

    largo_marca = max(
        8,
        int(
            distancia_y
            * LARGO_MARCA_LATERAL
        ),
    )

    for x in [
        centro_x - desplazamiento,
        centro_x + desplazamiento,
    ]:
        cv2.line(
            mascara,
            (
                x,
                centro_y - largo_marca,
            ),
            (
                x,
                centro_y + largo_marca,
            ),
            255,
            GROSOR_CROSSHAIR,
            cv2.LINE_AA,
        )

    return mascara


def crear_mascara_crosshair(
    frame,
    zona_crosshair,
    apertura_anterior=None,
):
    evidencia = obtener_evidencia_crosshair(
        frame,
        zona_crosshair,
    )

    (
        apertura,
        esquinas_detectadas,
        puntuacion,
    ) = estimar_apertura_crosshair(
        evidencia,
        apertura_anterior,
    )

    mascara = dibujar_crosshair_completo(
        evidencia.shape,
        apertura,
    )

    mascara = expandir(
        mascara,
        MARGEN_CROSSHAIR,
    )

    residuo = cv2.bitwise_and(
        evidencia,
        expandir(
            mascara,
            RADIO_RESIDUAL_CROSSHAIR,
        ),
    )

    mascara = cv2.bitwise_or(
        mascara,
        expandir(
            residuo,
            5,
        ),
    )

    mascara = cv2.bitwise_and(
        mascara,
        zona_crosshair,
    )

    return (
        mascara,
        apertura,
        esquinas_detectadas,
        puntuacion,
    )

# Zonas Circulares

In [8]:
def obtener_zonas_circulares(
    forma,
):
    zona_izquierda = crear_zona(
        forma,
        [
            (
                0.405,
                0.828,
                0.500,
                0.995,
            ),
        ],
    )

    zona_derecha = crear_zona(
        forma,
        [
            (
                0.495,
                0.828,
                0.590,
                0.995,
            ),
        ],
    )

    return [
        zona_izquierda,
        zona_derecha,
    ]

In [9]:
def ajustar_circulo(
    mascara_arco,
):
    posiciones_y, posiciones_x = np.where(
        mascara_arco > 0
    )

    if len(posiciones_x) < MINIMO_PIXELES_ARCO:
        return None

    x = posiciones_x.astype(
        np.float64
    )

    y = posiciones_y.astype(
        np.float64
    )

    matriz = np.column_stack([
        2.0 * x,
        2.0 * y,
        np.ones_like(x),
    ])

    objetivo = (
        x * x
        + y * y
    )

    try:
        solucion, _, _, _ = np.linalg.lstsq(
            matriz,
            objetivo,
            rcond=None,
        )

    except np.linalg.LinAlgError:
        return None

    centro_x = float(
        solucion[0]
    )

    centro_y = float(
        solucion[1]
    )

    termino = float(
        solucion[2]
    )

    radio_cuadrado = (
        centro_x * centro_x
        + centro_y * centro_y
        + termino
    )

    if radio_cuadrado <= 0:
        return None

    radio = float(
        np.sqrt(
            radio_cuadrado
        )
    )

    return (
        centro_x,
        centro_y,
        radio,
    )


def detectar_circulo_indicador(
    frame,
    zona_circulo,
    parametros_anteriores=None,
):
    medio = detectar_verde(
        frame,
        VERDE_MEDIO,
    )

    tenue = detectar_verde(
        frame,
        VERDE_TENUE,
    )

    señal = cv2.bitwise_and(
        cv2.bitwise_or(
            medio,
            tenue,
        ),
        zona_circulo,
    )

    señal = cv2.morphologyEx(
        señal,
        cv2.MORPH_CLOSE,
        cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (5, 5),
        ),
        iterations=1,
    )

    posiciones_y, posiciones_x = np.where(
        zona_circulo > 0
    )

    if len(posiciones_x) == 0:
        return parametros_anteriores

    izquierda = int(
        posiciones_x.min()
    )

    derecha = int(
        posiciones_x.max()
    ) + 1

    superior = int(
        posiciones_y.min()
    )

    inferior = int(
        posiciones_y.max()
    ) + 1

    region = señal[
        superior:inferior,
        izquierda:derecha,
    ]

    ajuste_local = ajustar_circulo(
        region,
    )

    candidatos = []

    if ajuste_local is not None:
        centro_x, centro_y, radio = (
            ajuste_local
        )

        candidatos.append((
            centro_x + izquierda,
            centro_y + superior,
            radio,
        ))

    suavizada = cv2.GaussianBlur(
        region,
        (5, 5),
        1.2,
    )

    ancho_total = señal.shape[1]

    circulos_hough = cv2.HoughCircles(
        suavizada,
        cv2.HOUGH_GRADIENT,
        dp=1.0,
        minDist=25,
        param1=70,
        param2=7,
        minRadius=max(
            10,
            int(
                ancho_total
                * RADIO_CIRCULO_MINIMO
            ),
        ),
        maxRadius=max(
            20,
            int(
                ancho_total
                * RADIO_CIRCULO_MAXIMO
            ),
        ),
    )

    if circulos_hough is not None:
        for x, y, radio in np.round(
            circulos_hough[0]
        ).astype(int):
            candidatos.append((
                x + izquierda,
                y + superior,
                radio,
            ))

    if not candidatos:
        return parametros_anteriores

    centro_zona_x = float(
        posiciones_x.mean()
    )

    centro_zona_y = float(
        posiciones_y.mean()
    )

    radio_minimo = (
        ancho_total
        * RADIO_CIRCULO_MINIMO
    )

    radio_maximo = (
        ancho_total
        * RADIO_CIRCULO_MAXIMO
    )

    candidatos_validos = []

    for centro_x, centro_y, radio in candidatos:
        if (
            radio < radio_minimo
            or radio > radio_maximo
        ):
            continue

        distancia_centro = np.hypot(
            centro_x - centro_zona_x,
            centro_y - centro_zona_y,
        )

        candidatos_validos.append((
            distancia_centro,
            centro_x,
            centro_y,
            radio,
        ))

    if not candidatos_validos:
        return parametros_anteriores

    candidatos_validos.sort(
        key=lambda candidato: candidato[0]
    )

    _, centro_x, centro_y, radio = (
        candidatos_validos[0]
    )

    actuales = (
        int(round(centro_x)),
        int(round(centro_y)),
        int(round(radio)),
    )

    if parametros_anteriores is None:
        return actuales

    return tuple(
        int(round(
            PESO_CIRCULO_ANTERIOR
            * anterior
            + (
                1.0
                - PESO_CIRCULO_ANTERIOR
            )
            * actual
        ))
        for anterior, actual in zip(
            parametros_anteriores,
            actuales,
        )
    )


def reconstruir_indicador_circular(
    frame,
    zona_circulo,
    parametros,
):
    mascara = np.zeros(
        zona_circulo.shape,
        dtype=np.uint8,
    )

    if parametros is None:
        return mascara

    centro_x, centro_y, radio = (
        parametros
    )

    radio_final = (
        radio
        + MARGEN_CIRCULO
    )

    cv2.circle(
        mascara,
        (
            centro_x,
            centro_y,
        ),
        radio_final,
        255,
        GROSOR_CIRCULO,
        cv2.LINE_AA,
    )

    señal = cv2.bitwise_and(
        cv2.bitwise_or(
            detectar_verde(
                frame,
                VERDE_MEDIO,
            ),
            detectar_verde(
                frame,
                VERDE_TENUE,
            ),
        ),
        zona_circulo,
    )

    lineas_conectadas = cv2.bitwise_and(
        señal,
        expandir(
            mascara,
            RADIO_LINEAS_CIRCULO,
        ),
    )

    mascara = cv2.bitwise_or(
        mascara,
        expandir(
            lineas_conectadas,
            DILATACION_LINEAS_CIRCULO,
        ),
    )

    return cv2.bitwise_and(
        mascara,
        expandir(
            zona_circulo,
            12,
        ),
    )

## Máscaras del HUD

Crea el atlas, completa los círculos y guarda una máscara por frame.


In [10]:
def crear_atlas_texto(frames):
    cantidad = min(MUESTRAS_ATLAS, len(frames))
    indices = np.linspace(0, len(frames) - 1, cantidad).astype(int)
    acumulador = None

    for indice in indices:
        frame = cv2.imread(str(frames[indice]))
        fuerte = detectar_verde(frame, VERDE_FUERTE)
        zona_texto, _, _, _ = obtener_zonas(fuerte.shape)
        muestra = cv2.bitwise_and(fuerte, zona_texto)

        if acumulador is None:
            acumulador = np.zeros(muestra.shape, dtype=np.float32)

        acumulador += (muestra > 0).astype(np.float32)

    return ((acumulador / cantidad) >= UMBRAL_ATLAS).astype(np.uint8) * 255


def crear_mascaras(
    base,
):
    base = Path(
        base
    )

    frames = sorted(
        (
            base
            / "frames"
        ).glob(
            "*.png"
        )
    )

    if not frames:
        raise RuntimeError(
            "No hay frames para procesar"
        )

    for nombre in [
        "masks",
        "vistas",
        "diagnosticos",
    ]:
        carpeta = (
            base
            / nombre
        )

        carpeta.mkdir(
            parents=True,
            exist_ok=True,
        )

        if LIMPIAR_ANTERIORES:
            limpiar_png(
                carpeta
            )

    atlas = crear_atlas_texto(
        frames
    )

    guardar_png(
        base
        / "atlas_texto.png",
        atlas,
    )

    apertura = None

    parametros_circulos = [
        None,
        None,
    ]

    for indice, ruta in enumerate(
        frames
    ):
        frame = cv2.imread(
            str(ruta)
        )

        if frame is None:
            raise RuntimeError(
                f"No se pudo leer {ruta}"
            )

        fuerte = detectar_verde(
            frame,
            VERDE_FUERTE,
        )

        medio = detectar_verde(
            frame,
            VERDE_MEDIO,
        )

        tenue = detectar_verde(
            frame,
            VERDE_TENUE,
        )

        (
            zona_texto,
            zona_brujula,
            zona_crosshair,
            zona_inferior,
        ) = obtener_zonas(
            fuerte.shape
        )

        soporte_texto = cv2.bitwise_and(
            expandir(
                atlas,
                RADIO_ATLAS,
            ),
            zona_texto,
        )

        mascara_texto = cv2.bitwise_and(
            cv2.bitwise_or(
                fuerte,
                medio,
            ),
            soporte_texto,
        )

        mascara_texto = expandir(
            mascara_texto,
            3,
        )

        (
            mascara_crosshair,
            apertura,
            esquinas_detectadas,
            puntuacion_crosshair,
        ) = crear_mascara_crosshair(
            frame,
            zona_crosshair,
            apertura,
        )

        mascara_brujula = cv2.bitwise_and(
            cv2.bitwise_or(
                fuerte,
                medio,
            ),
            zona_brujula,
        )

        mascara_brujula = cv2.morphologyEx(
            mascara_brujula,
            cv2.MORPH_CLOSE,
            cv2.getStructuringElement(
                cv2.MORPH_RECT,
                (3, 13),
            ),
            iterations=1,
        )

        mascara_brujula = expandir(
            mascara_brujula,
            4,
        )

        mascara_inferior = cv2.bitwise_and(
            cv2.bitwise_or(
                fuerte,
                medio,
            ),
            zona_inferior,
        )

        mascara_inferior = cv2.morphologyEx(
            mascara_inferior,
            cv2.MORPH_CLOSE,
            cv2.getStructuringElement(
                cv2.MORPH_RECT,
                (13, 3),
            ),
            iterations=1,
        )

        mascara_inferior = expandir(
            mascara_inferior,
            4,
        )

        zonas_circulares = obtener_zonas_circulares(
            fuerte.shape
        )

        mascaras_circulares = []

        for indice_circulo, zona_circulo in enumerate(
            zonas_circulares
        ):
            parametros_circulos[
                indice_circulo
            ] = detectar_circulo_indicador(
                frame,
                zona_circulo,
                parametros_circulos[
                    indice_circulo
                ],
            )

            mascara_circular = reconstruir_indicador_circular(
                frame,
                zona_circulo,
                parametros_circulos[
                    indice_circulo
                ],
            )

            mascaras_circulares.append(
                mascara_circular
            )

        mascara = cv2.bitwise_or(
            mascara_texto,
            mascara_crosshair,
        )

        mascara = cv2.bitwise_or(
            mascara,
            mascara_brujula,
        )

        mascara = cv2.bitwise_or(
            mascara,
            mascara_inferior,
        )

        for mascara_circular in mascaras_circulares:
            mascara = cv2.bitwise_or(
                mascara,
                mascara_circular,
            )

        zona_hud = cv2.bitwise_or(
            zona_texto,
            zona_brujula,
        )

        zona_hud = cv2.bitwise_or(
            zona_hud,
            zona_crosshair,
        )

        zona_hud = cv2.bitwise_or(
            zona_hud,
            zona_inferior,
        )

        residuo = cv2.bitwise_and(
            tenue,
            zona_hud,
        )

        residuo = cv2.bitwise_and(
            residuo,
            expandir(
                mascara,
                RADIO_RESIDUAL_HUD,
            ),
        )

        mascara = cv2.bitwise_or(
            mascara,
            expandir(
                residuo,
                4,
            ),
        )

        mascara = (
            mascara > 0
        ).astype(
            np.uint8
        ) * 255

        guardar_png(
            base
            / "masks"
            / ruta.name,
            mascara,
        )

        if GUARDAR_VISTAS:
            rojo = np.zeros_like(
                frame
            )

            rojo[:, :, 2] = 255

            mezcla = cv2.addWeighted(
                frame,
                0.60,
                rojo,
                0.40,
                0,
            )

            vista = frame.copy()

            vista[
                mascara > 0
            ] = mezcla[
                mascara > 0
            ]

            guardar_png(
                base
                / "vistas"
                / ruta.name,
                vista,
            )

        if GUARDAR_DIAGNOSTICOS:
            diagnostico = frame.copy()

            plantilla = dibujar_crosshair_completo(
                mascara.shape,
                apertura,
            )

            diagnostico[
                plantilla > 0
            ] = (
                255,
                0,
                255,
            )

            for parametros in parametros_circulos:
                if parametros is None:
                    continue

                centro_x, centro_y, radio = (
                    parametros
                )

                cv2.circle(
                    diagnostico,
                    (
                        centro_x,
                        centro_y,
                    ),
                    radio,
                    (
                        0,
                        255,
                        255,
                    ),
                    2,
                    cv2.LINE_AA,
                )

                cv2.circle(
                    diagnostico,
                    (
                        centro_x,
                        centro_y,
                    ),
                    4,
                    (
                        255,
                        255,
                        0,
                    ),
                    -1,
                    cv2.LINE_AA,
                )

            texto_diagnostico = (
                f"dx={apertura[0]} "
                f"dy={apertura[1]} "
                f"esquinas={esquinas_detectadas} "
                f"puntaje={puntuacion_crosshair:.1f}"
            )

            cv2.putText(
                diagnostico,
                texto_diagnostico,
                (
                    20,
                    30,
                ),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.65,
                (
                    255,
                    255,
                    0,
                ),
                2,
                cv2.LINE_AA,
            )

            guardar_png(
                base
                / "diagnosticos"
                / ruta.name,
                diagnostico,
            )

        if (
            indice + 1
        ) % 100 == 0:
            print(
                f"Máscaras: "
                f"{indice + 1}/"
                f"{len(frames)}"
            )

    print(
        f"Máscaras creadas: {len(frames)}"
    )

    return base

## Plan de Bloques

Calcula cuántos bloques habrá. El solapamiento total se divide entre ambos lados.


In [11]:
def calcular_bloques(
    total_frames,
    frames_principales=FRAMES_PRINCIPALES_POR_BLOQUE,
    solapamiento_total=SOLAPAMIENTO_TOTAL,
):
    contexto_izquierdo = solapamiento_total // 2
    contexto_derecho = solapamiento_total - contexto_izquierdo
    cantidad = math.ceil(total_frames / frames_principales)
    bloques = []

    for numero in range(cantidad):
        inicio = numero * frames_principales
        fin = min(total_frames, inicio + frames_principales)
        inicio_contexto = max(0, inicio - contexto_izquierdo)
        fin_contexto = min(total_frames, fin + contexto_derecho)

        bloques.append({
            "numero": numero + 1,
            "inicio": inicio,
            "fin": fin,
            "inicio_contexto": inicio_contexto,
            "fin_contexto": fin_contexto,
            "principales": fin - inicio,
            "total": fin_contexto - inicio_contexto,
        })

    return bloques


def mostrar_bloques(
    base,
    frames_principales=FRAMES_PRINCIPALES_POR_BLOQUE,
    solapamiento_total=SOLAPAMIENTO_TOTAL,
):
    base = Path(base)
    frames = sorted((base / "frames").glob("*.png"))
    bloques = calcular_bloques(
        len(frames),
        frames_principales,
        solapamiento_total,
    )

    print(f"Frames disponibles: {len(frames)}")
    print(f"Bloques necesarios: {len(bloques)}")
    print(f"Solapamiento total: {solapamiento_total}")
    print()

    for bloque in bloques:
        print(
            f"Bloque {bloque['numero']:02d}: "
            f"{bloque['principales']} principales, "
            f"{bloque['total']} para procesar"
        )

    return bloques


In [12]:
calcular_bloques(3330, 100, 20)

[{'numero': 1,
  'inicio': 0,
  'fin': 100,
  'inicio_contexto': 0,
  'fin_contexto': 110,
  'principales': 100,
  'total': 110},
 {'numero': 2,
  'inicio': 100,
  'fin': 200,
  'inicio_contexto': 90,
  'fin_contexto': 210,
  'principales': 100,
  'total': 120},
 {'numero': 3,
  'inicio': 200,
  'fin': 300,
  'inicio_contexto': 190,
  'fin_contexto': 310,
  'principales': 100,
  'total': 120},
 {'numero': 4,
  'inicio': 300,
  'fin': 400,
  'inicio_contexto': 290,
  'fin_contexto': 410,
  'principales': 100,
  'total': 120},
 {'numero': 5,
  'inicio': 400,
  'fin': 500,
  'inicio_contexto': 390,
  'fin_contexto': 510,
  'principales': 100,
  'total': 120},
 {'numero': 6,
  'inicio': 500,
  'fin': 600,
  'inicio_contexto': 490,
  'fin_contexto': 610,
  'principales': 100,
  'total': 120},
 {'numero': 7,
  'inicio': 600,
  'fin': 700,
  'inicio_contexto': 590,
  'fin_contexto': 710,
  'principales': 100,
  'total': 120},
 {'numero': 8,
  'inicio': 700,
  'fin': 800,
  'inicio_contexto': 

## Preparar un Bloque

Copia los frames y las máscaras de un bloque concreto.


In [13]:
def preparar_bloque(
    base,
    numero_bloque,
    frames_principales=FRAMES_PRINCIPALES_POR_BLOQUE,
    solapamiento_total=SOLAPAMIENTO_TOTAL,
):
    base = Path(base)

    frames = sorted(
        (base / "frames").glob("*.png")
    )

    mascaras = sorted(
        (base / "masks").glob("*.png")
    )

    vistas = sorted(
        (base / "vistas").glob("*.png")
    )

    diagnosticos = sorted(
        (base / "diagnosticos").glob("*.png")
    )

    if len(frames) != len(mascaras):
        raise RuntimeError(
            "Las cantidades de frames y máscaras no coinciden"
        )

    nombres_frames = [
        archivo.name
        for archivo in frames
    ]

    nombres_mascaras = [
        archivo.name
        for archivo in mascaras
    ]

    if nombres_frames != nombres_mascaras:
        raise RuntimeError(
            "Los nombres de frames y máscaras no coinciden"
        )

    copiar_vistas = (
        len(vistas) == len(frames)
        and [
            archivo.name
            for archivo in vistas
        ] == nombres_frames
    )

    copiar_diagnosticos = (
        len(diagnosticos) == len(frames)
        and [
            archivo.name
            for archivo in diagnosticos
        ] == nombres_frames
    )

    bloques = calcular_bloques(
        total_frames=len(frames),
        frames_principales=frames_principales,
        solapamiento_total=solapamiento_total,
    )

    if (
        numero_bloque < 1
        or numero_bloque > len(bloques)
    ):
        raise ValueError(
            f"El bloque debe estar entre 1 y {len(bloques)}"
        )

    datos = bloques[
        numero_bloque - 1
    ]

    carpeta = (
        base
        / f"bloque_{numero_bloque:02d}"
    )

    if carpeta.exists():
        shutil.rmtree(
            carpeta
        )

    subcarpetas = [
        "frames",
        "masks",
        "no_hud",
    ]

    if copiar_vistas:
        subcarpetas.append(
            "vistas"
        )

    if copiar_diagnosticos:
        subcarpetas.append(
            "diagnosticos"
        )

    for nombre in subcarpetas:
        (
            carpeta
            / nombre
        ).mkdir(
            parents=True,
            exist_ok=True,
        )

    for indice in range(
        datos["inicio_contexto"],
        datos["fin_contexto"],
    ):
        nombre = frames[
            indice
        ].name

        shutil.copy2(
            frames[indice],
            carpeta
            / "frames"
            / nombre,
        )

        shutil.copy2(
            mascaras[indice],
            carpeta
            / "masks"
            / nombre,
        )

        if copiar_vistas:
            shutil.copy2(
                vistas[indice],
                carpeta
                / "vistas"
                / nombre,
            )

        if copiar_diagnosticos:
            shutil.copy2(
                diagnosticos[indice],
                carpeta
                / "diagnosticos"
                / nombre,
            )

    nombres_finales = [
        frames[indice].name
        for indice in range(
            datos["inicio"],
            datos["fin"],
        )
    ]

    (
        carpeta
        / "frames_finales.txt"
    ).write_text(
        "\n".join(
            nombres_finales
        )
        + "\n",
        encoding="utf-8",
    )

    print()
    print(
        f"Bloque preparado: {numero_bloque:02d}"
    )
    print(
        f"Frames principales: {datos['principales']}"
    )
    print(
        f"Frames de contexto: "
        f"{datos['total'] - datos['principales']}"
    )
    print(
        f"Frames para procesar: {datos['total']}"
    )
    print(
        f"Vistas copiadas: {'Sí' if copiar_vistas else 'No'}"
    )
    print(
        f"Diagnósticos copiados: "
        f"{'Sí' if copiar_diagnosticos else 'No'}"
    )
    print(
        f"Carpeta: {carpeta}"
    )

    return carpeta

In [14]:
preparar_bloque(r'Videos\video30min-11to22', 1, 100, 20)


Bloque preparado: 01
Frames principales: 100
Frames de contexto: 10
Frames para procesar: 110
Vistas copiadas: Sí
Diagnósticos copiados: Sí
Carpeta: Videos\video30min-11to22\bloque_01


WindowsPath('Videos/video30min-11to22/bloque_01')

## Preparar Todos los Bloques

Crea automáticamente todas las carpetas listas para ProPainter.


In [9]:
def preparar_todos_los_bloques(
    base,
    frames_principales=FRAMES_PRINCIPALES_POR_BLOQUE,
    solapamiento_total=SOLAPAMIENTO_TOTAL,
):
    bloques = mostrar_bloques(
        base,
        frames_principales,
        solapamiento_total,
    )

    carpetas = []

    for bloque in bloques:
        carpeta = preparar_bloque(
            base,
            bloque["numero"],
            frames_principales,
            solapamiento_total,
        )
        carpetas.append(carpeta)

    return carpetas


## Ejecutar el Pipeline

Extrae los frames, crea las máscaras y deja todos los bloques listos.


In [ ]:
def procesar_video(
    video=VIDEO,
    salida=SALIDA,
    frames_principales=FRAMES_PRINCIPALES_POR_BLOQUE,
    solapamiento_total=SOLAPAMIENTO_TOTAL,
):
    base = extraer_frames(video=video, salida=salida)
    crear_mascaras(base)
    preparar_todos_los_bloques(
        base,
        frames_principales,
        solapamiento_total,
    )

    print("Pipeline terminado")
    return base


base_resultado = procesar_video()
